In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

# ============================================================
#  Brain Tumor Federated Dataset — Data Augmentation
# ============================================================
# Structure preserved:
#   <OUTPUT_ROOT>/BrainTumor/client_X/class_name/*.jpg
#
# Strategy:
#   • client_0, client_1, client_2  → augmented  (training clients)
#   • client_3                      → copied as-is via copytree (test set)
#   • Each image gets N augmented copies, each with a DIFFERENT
#     transform chosen randomly and independently per image
#   • Normal classes : 2 augments per image  →  x3 images
#   • no_tumor class : 3 augments per image  →  x4 images  (re-balance)
#
#
#
# Available transforms (one is picked at random per augmented copy):
#   1.  Horizontal flip
#   2.  Vertical flip
#   3.  Rotation +-20 degrees
#   4.  Affine (translate + scale + shear)
#   5.  Random crop and resize (zoom simulation)
#   6.  Gaussian blur (scanner noise)
#   7.  Mild brightness + contrast
#   8.  Horizontal flip + rotation
#   9.  Affine + blur
#   10. Crop + contrast
#

In [3]:
!pip install torchvision Pillow tqdm -q

In [4]:
import os
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as T

In [5]:
INPUT_ROOT  = "/content/drive/MyDrive/FACMIC/data"
OUTPUT_ROOT = "/content/drive/MyDrive/FACMIC/data_augmented"

In [6]:
CLIENTS_TO_AUGMENT = ["client_0", "client_1", "client_2"]
CLIENTS_COPY_ONLY  = ["client_3"]   # test set — never augmented

CLASSES        = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
MINORITY_CLASS = "no_tumor"

N_AUG_NORMAL   = 2   # augmented copies per image for normal classes
N_AUG_MINORITY = 3   # augmented copies per image for no_tumor

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}
SEED           = 42

In [7]:
def build_transforms():
    """
    Returns a dict of named MRI-appropriate transforms.
    Each entry is (name, transform) so we can log which transform
    was applied to each image and verify the distribution is spread.
    No hue or saturation changes — MRI has no colour information.
    """
    return {
        "hflip": T.RandomHorizontalFlip(p=1.0),

        "vflip": T.RandomVerticalFlip(p=1.0),

        "rotate": T.RandomRotation(degrees=20, fill=0),

        "affine": T.RandomAffine(
            degrees=15,
            translate=(0.10, 0.10),
            scale=(0.85, 1.15),
            shear=10,
            fill=0,
        ),

        "zoom": T.RandomResizedCrop(
            size=(512, 512),
            scale=(0.75, 1.0),
            ratio=(0.9, 1.1),
            antialias=True,
        ),

        "blur": T.GaussianBlur(kernel_size=5, sigma=(0.5, 2.0)),

        "jitter": T.ColorJitter(brightness=0.15, contrast=0.15),

        "hflip_rotate": T.Compose([
            T.RandomHorizontalFlip(p=1.0),
            T.RandomRotation(degrees=15, fill=0),
        ]),

        "affine_blur": T.Compose([
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), fill=0),
            T.GaussianBlur(kernel_size=3, sigma=(0.3, 1.5)),
        ]),

        "zoom_jitter": T.Compose([
            T.RandomResizedCrop(size=(512, 512), scale=(0.80, 1.0), antialias=True),
            T.ColorJitter(contrast=0.20),
        ]),
    }

In [8]:
def pick_and_apply(
    img:        Image.Image,
    transforms: dict,
    used_names: set,
    rng:        random.Random,
    img_seed:   int,
) -> tuple:
    """
    Pick one transform at random from the ones NOT yet used for this image,
    apply it, and return (augmented_image, transform_name).

    This guarantees that the N copies of the same image each get
    a DIFFERENT transform. Once all transforms are used, the pool resets.
    """
    available = [k for k in transforms if k not in used_names]
    if not available:
        available = list(transforms.keys())   # reset if pool exhausted

    name = rng.choice(available)
    tf   = transforms[name]

    # Seed PyTorch so stochastic transforms are reproducible
    torch.manual_seed(img_seed)
    aug = tf(img)

    return aug, name



In [9]:
def augment_dataset():
    master_rng = random.Random(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    transforms = build_transforms()
    tf_names   = list(transforms.keys())

    brain_in  = Path(INPUT_ROOT)  / "BrainTumor"
    brain_out = Path(OUTPUT_ROOT) / "BrainTumor"

    total_original  = 0
    total_augmented = 0

    # Track how often each transform is used across the whole dataset
    transform_usage = defaultdict(int)

    print("=" * 68)
    print("  BrainTumor Federated Dataset — Augmentation")
    print("=" * 68)
    print(f"  Input       : {brain_in}")
    print(f"  Output      : {brain_out}")
    print(f"  Augmented   : {CLIENTS_TO_AUGMENT}")
    print(f"  Copied only : {CLIENTS_COPY_ONLY}  (test set- no augmentation)")
    print(f"  Normal      : {N_AUG_NORMAL} random augment(s) per image")
    print(f"  {MINORITY_CLASS} : {N_AUG_MINORITY} random augment(s) per image")
    print(f"  Available transforms ({len(tf_names)}): {tf_names}")
    print("=" * 68)

    # ── Step 1: Copy client_3 as-is using copytree ────────────
    for client in CLIENTS_COPY_ONLY:
        src = brain_in  / client
        dst = brain_out / client
        if not src.exists():
            print(f"\n[SKIP] {client} not found.")
            continue
        if dst.exists():
            shutil.rmtree(dst)
        print(f"\n[COPY   ] {client}  →  copying entire folder ...")
        shutil.copytree(src, dst)
        n = sum(1 for f in dst.rglob("*") if f.suffix.lower() in IMAGE_SUFFIXES)
        print(f"  Done — {n} images copied.")
        total_original += n

    # ── Step 2: Augment training clients ─────────────────────
    for client in CLIENTS_TO_AUGMENT:
        client_in  = brain_in  / client
        client_out = brain_out / client

        if not client_in.exists():
            print(f"\n[SKIP] {client} not found.")
            continue

        print(f"\n[AUGMENT] {client}")

        for cls_name in CLASSES:
            cls_in  = client_in  / cls_name
            cls_out = client_out / cls_name

            if not cls_in.exists():
                print(f"  [SKIP] {cls_name} not found.")
                continue

            cls_out.mkdir(parents=True, exist_ok=True)

            files = sorted(
                p for p in cls_in.iterdir()
                if p.suffix.lower() in IMAGE_SUFFIXES
            )

            n_aug     = N_AUG_MINORITY if cls_name == MINORITY_CLASS else N_AUG_NORMAL
            total_out = len(files) * (1 + n_aug)

            print(f"  {cls_name:<22s}: {len(files):>4} originals"
                  f"  →  {total_out:>5} total"
                  f"  ({n_aug} random augments each)")

            for fpath in tqdm(files, desc=f"    {cls_name}", leave=False):

                # Always copy the original unchanged
                shutil.copy2(fpath, cls_out / fpath.name)
                total_original += 1

                # Load once, generate n_aug differently-transformed copies
                img = Image.open(fpath).convert("RGB")

                # Track which transforms have been used for THIS image
                # so each copy gets a different one
                used_for_this_image = set()

                for k in range(n_aug):
                    # Unique seed per (file, augment_index) for reproducibility
                    img_seed = abs(hash(fpath.name)) + k * 9973

                    aug, tf_name = pick_and_apply(
                        img        = img,
                        transforms = transforms,
                        used_names = used_for_this_image,
                        rng        = master_rng,
                        img_seed   = img_seed,
                    )

                    used_for_this_image.add(tf_name)
                    transform_usage[tf_name] += 1

                    aug_name = f"{fpath.stem}_aug{k}_{tf_name}{fpath.suffix}"
                    aug.save(cls_out / aug_name, quality=95)
                    total_augmented += 1

    # ── Summary ───────────────────────────────────────────────
    print("\n" + "=" * 68)
    print("  Augmentation complete!")
    print("=" * 68)
    print(f"  Original images  : {total_original}")
    print(f"  Augmented images : {total_augmented}")
    print(f"  Grand total      : {total_original + total_augmented}")
    print(f"  Saved to         : {brain_out}")

    # Transform usage distribution
    print(f"\n  Transform usage distribution ({total_augmented} augmented images):")
    print(f"  {'Transform':<20s} {'Count':>6}  {'Share':>7}")
    print("  " + "-" * 38)
    for name in tf_names:
        count = transform_usage[name]
        share = 100.0 * count / total_augmented if total_augmented > 0 else 0
        bar   = "█" * int(share / 2)
        print(f"  {name:<20s} {count:>6}   {share:>5.1f}%  {bar}")

    # Per-client per-class count table
    print(f"\n  {'':12s} {'glioma':>8} {'meningioma':>12} "
          f"{'no_tumor':>10} {'pituitary':>11} {'total':>7}")
    print("  " + "-" * 56)
    grand_total = 0
    for client in CLIENTS_TO_AUGMENT + CLIENTS_COPY_ONLY:
        counts = []
        for cls_name in CLASSES:
            cls_out = brain_out / client / cls_name
            if cls_out.exists():
                n = sum(
                    1 for f in cls_out.iterdir()
                    if f.suffix.lower() in IMAGE_SUFFIXES
                )
            else:
                n = 0
            counts.append(n)
        row = sum(counts)
        grand_total += row
        print(f"  {client:<12s} {counts[0]:>8} {counts[1]:>12} "
              f"{counts[2]:>10} {counts[3]:>11} {row:>7}")
    print("  " + "-" * 56)
    print(f"  {'TOTAL':<12s} {grand_total:>49}")


In [10]:
augment_dataset()

  BrainTumor Federated Dataset — Augmentation
  Input       : /content/drive/MyDrive/FACMIC/data/BrainTumor
  Output      : /content/drive/MyDrive/FACMIC/data_augmented/BrainTumor
  Augmented   : ['client_0', 'client_1', 'client_2']
  Copied only : ['client_3']  (test set- no augmentation)
  Normal      : 2 random augment(s) per image
  no_tumor : 3 random augment(s) per image
  Available transforms (10): ['hflip', 'vflip', 'rotate', 'affine', 'zoom', 'blur', 'jitter', 'hflip_rotate', 'affine_blur', 'zoom_jitter']

[COPY   ] client_3  →  copying entire folder ...
  Done — 394 images copied.

[AUGMENT] client_0
  glioma_tumor          :  424 originals  →   1272 total  (2 random augments each)


  meningioma_tumor      :  424 originals  →   1272 total  (2 random augments each)


  no_tumor              :  203 originals  →    812 total  (3 random augments each)


  pituitary_tumor       :  425 originals  →   1275 total  (2 random augments each)



[AUGMENT] client_1
  glioma_tumor          :  124 originals  →    372 total  (2 random augments each)


...


[AUGMENT] client_2
  glioma_tumor          :  273 originals  →    819 total  (2 random augments each)


  meningioma_tumor      :  272 originals  →    816 total  (2 random augments each)


  no_tumor              :  130 originals  →    520 total  (3 random augments each)


  pituitary_tumor       :  274 originals  →    822 total  (2 random augments each)



  Augmentation complete!
  Original images  : 3244
  Augmented images : 6091
  Grand total      : 9335
  Saved to         : /content/drive/MyDrive/FACMIC/data_augmented/BrainTumor

  Transform usage distribution (6091 augmented images):
  Transform             Count    Share
  --------------------------------------
  hflip                   570     9.4%  ████
  vflip                   626    10.3%  █████
  rotate                  609    10.0%  ████
  affine                  590     9.7%  ████
  zoom                    596     9.8%  ████
  blur                    613    10.1%  █████
  jitter                  602     9.9%  ████
  hflip_rotate            623    10.2%  █████
  affine_blur             625    10.3%  █████
  zoom_jitter             637    10.5%  █████

                 glioma   meningioma   no_tumor   pituitary   total
  --------------------------------------------------------
  client_0         1272         1272        812        1275    4631
  client_1          372        